In [ ]:
!pip install transformers datasets torch numpy tqdm

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch
import numpy as np
from tqdm import tqdm
import json

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = AutoModel.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
).to(device)

encoder.eval()

tokenizer = AutoTokenizer.from_pretrained(
    "JayShah07/tinybert-dual-classifier"
)

In [ ]:
dataset = load_dataset("JayShah07/reporting_final_dataset")
train_ds = dataset["train"]

print("Train size:", len(train_ds))

In [ ]:
embeddings = []
module_confidences = []
date_confidences = []

softmax = torch.nn.Softmax(dim=-1)

# ---------------- LOOP OVER TRAINING DATA ----------------
for sample in tqdm(train_ds):
    text = sample["query"]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        # ---------------- CLS EMBEDDING ----------------
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, hidden_size)
        embeddings.append(cls_embedding.cpu().numpy()[0])

        # ---------------- MODULE & DATE LOGITS ----------------
        # In real scenario: pass cls_embedding to your trained linear heads
        # Here, we mock logits using slices of CLS embedding
        module_logits = cls_embedding[:, :6]  # first 6 dims -> module logits
        date_logits = cls_embedding[:, 6:13]  # next 7 dims -> date logits

        module_probs = softmax(module_logits)
        date_probs = softmax(date_logits)

        module_confidences.append(module_probs.max().item())
        date_confidences.append(date_probs.max().item())

# ---------------- CONVERT TO NUMPY ----------------
embeddings = np.array(embeddings)  # (num_samples, hidden_size)
module_confidences = np.array(module_confidences)
date_confidences = np.array(date_confidences)

print("Embeddings shape:", embeddings.shape)
print("Module confidence samples:", module_confidences[:5])
print("Date confidence samples:", date_confidences[:5])

# ---------------- COMPUTE EMBEDDING BASELINE ----------------
embedding_baseline = embeddings.mean(axis=0)
embedding_cov = np.cov(embeddings.T)

# ---------------- SAVE BASELINES ----------------
np.save("embedding_baseline.npy", embedding_baseline)
np.save("embedding_cov.npy", embedding_cov)
np.save("module_conf_baseline.npy", module_confidences)
np.save("date_conf_baseline.npy", date_confidences)

# ---------------- CREATE HISTOGRAMS FOR PSI ----------------
module_hist, module_bins = np.histogram(module_confidences, bins=10)
date_hist, date_bins = np.histogram(date_confidences, bins=10)

with open("module_conf_hist.json", "w") as f:
    json.dump({"hist": module_hist.tolist(), "bins": module_bins.tolist()}, f)

with open("date_conf_hist.json", "w") as f:
    json.dump({"hist": date_hist.tolist(), "bins": date_bins.tolist()}, f)

print("Baselines saved: embeddings, module PSI, date PSI")